In [1]:
# Python 3.10.11
# %pip install -r requirements.txt > /dev/null
from args import *
from utils import *

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, accuracy_score, recall_score
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.data import random_split

from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.preprocessing import label_binarize

from torchmetrics import AUROC
from torch.utils.tensorboard import SummaryWriter


# TODO(241225) 依赖导入
import pandas as pd


In [3]:
# TODO(241225) 导入label数据
label_df = pd.read_csv(sample_labels_file_path, index_col=0)

# 裁减样本数量，使得 0 - 1 样本数量一致
df_0 = label_df[label_df['label'] == 0]
df_1 = label_df[label_df['label'] == 1]
min_len = min(len(df_0), len(df_1))
new_df = pd.concat([df_0[:min_len], df_1[:min_len]], axis=0)

sample_key_list = label_df.index.to_list()
sample_key_list = new_df.index.to_list() # 均衡数据

print(f"label 样本数量: {len(sample_key_list)}")

# TODO(241225) 导入gene数据
# 根据 sample_key_list 为基准, 若模态数据中不存在 sample_key 则填充新数据
gene_array = load_gene_data_by_sample_key(sample_key_list).values
cnv_array = load_cnv_data_by_sample_key(sample_key_list).values

wsi_array = load_wsi_data_by_sample_key(sample_key_list)
report_array = load_report_data_by_sample_key(sample_key_list)

label_array = label_df.values
label_array = new_df.values # 均衡数据

_, gene_dim = gene_array.shape
_, cnv_dim = cnv_array.shape
_, wsi_dim = wsi_array.shape
_, report_dim = report_array.shape
_, label_dim = label_array.shape

print(f"""
gene 数据维度:   {gene_dim}
cnv 数据维度:    {cnv_dim}
wsi 数据维度:    {wsi_dim}
report 数据维度: {report_dim}
""")

batch_size = 32

all_dataset = MultiOmicsDataset(gene_array, cnv_array, report_array, wsi_array, label_array)

# 假设 all_dataset 是一个 Dataset 对象
train_len = int(len(all_dataset) * 0.8)  # 80% 的数据用作训练集
test_len = len(all_dataset) - train_len  # 剩余的数据用作验证集

# 使用 random_split 分割数据集
train_val_dataset, test_dataset = random_split(all_dataset, [train_len, test_len])

train_loader = DataLoader(all_dataset, batch_size=batch_size, shuffle=True, num_workers=3, drop_last=False)
val_loader = DataLoader(all_dataset, batch_size=batch_size, shuffle=False, num_workers=3, drop_last=False)

label 样本数量: 152

gene 数据维度:   2340
cnv 数据维度:    2340
wsi 数据维度:    2048
report 数据维度: 768



In [4]:
### 分割线

In [5]:

from torch import nn, optim

def report_loader_layer(pre_trained=False):

    if not pre_trained:
        return nn.Sequential(
            nn.Linear(768, 768),
            nn.Linear(768, 2),
        )



    model_file_path = pkg_dir_path.parent / "report/data/output/bert.pth"
    assert model_file_path.exists()

    model = torch.load(model_file_path).to(device)
    model = model.classifier

    # 冻结参数
    # 冻结第一层
    for param in model[0].parameters():
        param.requires_grad = False

    # 确认第一层的参数不需要梯度
    for name, param in model.named_parameters():
        if name.startswith('0.'):
            print(name, param.requires_grad)  # 应该输出False

    return model

report_loader_layer = report_loader_layer()

In [6]:

from torch import nn, optim

def wsi_loader_layer(pre_trained=False):

    if not pre_trained:
        return nn.Sequential(
            nn.Linear(2048, 2048),
            nn.Linear(2048, 2),
        )

    return model

wsi_loader_layer = wsi_loader_layer()

In [7]:

from torch import nn, optim

def gene_loader_layer(pre_trained=False):

    if not pre_trained:
        return nn.Sequential(
            nn.Linear(2340, 2340),
            nn.ReLU(),
            nn.Linear(2340, 2),
        )

    return model

gene_loader_layer = gene_loader_layer()

In [8]:
import torch
from torch import nn, optim
from torch.utils.data import DataLoader
from torchvision import models  # 如果需要使用预训练模型

# 假设我们使用一个预训练的ResNet作为特征提取器，并添加自定义层
class CustomModel(nn.Module):
    def __init__(self, base_model, num_classes):
        super(CustomModel, self).__init__()
        # 假设base_model期望的输入形状是[batch_size, 3, 224, 224]
        # 我们将第一层替换为一个全连接层
        self.base_model = base_model
        # self.custom_layer = nn.Sequential(
        #     nn.Linear(base_model.in_features, 256),
        #     nn.ReLU(),
        #     # nn.Dropout(0.5),
        #     nn.Linear(256, num_classes)
        # )

    def forward(self, x):
        x = self.base_model(x)
        # x = self.custom_layer(x)
        return x

# 加载预训练模型并替换最后一层
num_classes = 1  # 根据您的任务确定类别数
model = CustomModel(report_loader_layer, num_classes).to(device)
model = CustomModel(wsi_loader_layer, num_classes).to(device)
model = CustomModel(gene_loader_layer, num_classes).to(device)

# 定义损失函数和优化器
criterion = nn.CrossEntropyLoss()  # 适用于二分类问题
optimizer = optim.Adam(model.parameters(), lr=0.00001)

# 学习率调度器，可以在训练过程中调整学习率
# scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min')

/root/miniforge3/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/root/miniforge3/lib/python3.10/site-packages/torchvision/image.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(


In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


# 构造含交叉注意力机制的Transformer
class TransformerEncoderLayerWithCrossAttention(nn.Module):
    def __init__(self, d_model, nhead, dropout=0.1, dim_feedforward=2048,):
        super(TransformerEncoderLayerWithCrossAttention, self).__init__()
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout)
        self.cross_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout)

        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(dim_feedforward, d_model)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

    def forward(self, src, src_mask=None):

                # 自注意力机制
        src2 = self.self_attn(src, src, src, attn_mask=src_mask)[0]
        src = src + self.dropout1(src2)
        src = self.norm1(src)

        # 将输入序列平均拆分为4等分
        seq_len = src.size(1)
        seg_len = seq_len // 4
        seg1 = src[:, :seg_len, :]
        seg2 = src[:, seg_len:2*seg_len, :]
        seg3 = src[:, 2*seg_len:3*seg_len, :]
        seg4 = src[:, 3*seg_len:, :]

        # 交叉注意力机制
        seg1_cross, _ = self.cross_attn(seg1, seg2, seg2)
        seg2_cross, _ = self.cross_attn(seg2, seg3, seg3)
        seg3_cross, _ = self.cross_attn(seg3, seg4, seg4)
        seg4_cross, _ = self.cross_attn(seg4, seg1, seg1)

        # 合并交叉注意力结果
        src_cross = torch.cat([seg1_cross, seg2_cross, seg3_cross, seg4_cross], dim=1)
        src = src + self.dropout2(src_cross)
        src = self.norm2(src)

        # 线性层和残差连接
        src2 = self.linear2(self.dropout(F.relu(self.linear1(src))))
        src = src + self.dropout3(src2)
        src = self.norm3(src)

        return src

In [10]:
class TransformerEncoderWithCrossAttention(nn.Module):
    def __init__(self, encoder_layer, num_layers, norm=None):
        super(TransformerEncoderWithCrossAttention, self).__init__()
        self.layers = nn.ModuleList([encoder_layer for _ in range(num_layers)])
        self.num_layers = num_layers
        self.norm = norm

    def forward(self, src, mask=None):
        output = src

        for layer in self.layers:
            output = layer(output, src_mask=mask)

        if self.norm is not None:
            output = self.norm(output)

        return output

# 定义模型结构
class MultiOmicsModel(nn.Module):
    def __init__(self, dropout_prob=0.2):
        super(MultiOmicsModel, self).__init__()

        # 分割线
        global gene_dim, cnv_dim, report_dim, wsi_dim, label_dim

        same_all_feature_dim = 256
        self.shared_hidden_gene_cnv = nn.Linear(gene_dim + cnv_dim, gene_dim)
        self.hidden_gene = nn.Sequential(nn.Linear(gene_dim, same_all_feature_dim))
        self.hidden_cnv = nn.Sequential(nn.Linear(gene_dim, same_all_feature_dim))

        self.fc_report = nn.Linear(report_dim, same_all_feature_dim)
        self.fc_wsi = nn.Linear(wsi_dim, same_all_feature_dim)

        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout_prob)

        

        # 创建一个带有交叉注意力的Transformer编码器层
        # nn.MultiheadAttention(d_model, nhead, dropout=dropout) 
        d_model, nhead, dropout = same_all_feature_dim, 1, 0.1
        dim_feedforward = 32
        num_layers = 1
        encoder_layer = TransformerEncoderLayerWithCrossAttention(d_model, nhead, dropout, dim_feedforward)
        # 创建一个带有交叉注意力的Transformer编码器
        self.encoder = TransformerEncoderWithCrossAttention(encoder_layer, num_layers)

        self.flatten = nn.Flatten(start_dim=1)

        # 输出层
        self.lin = nn.Linear(same_all_feature_dim * 4, 2)


    def forward(self, gene_tensor, cnv_tensor, report_tensor, wsi_tensor):

        gene_cnv_feature = torch.concat([gene_tensor, cnv_tensor], dim=1)
        gene_cnv_feature = self.relu(self.shared_hidden_gene_cnv(gene_cnv_feature))
        gene_cnv_feature = self.dropout(gene_cnv_feature)

        report_feature = self.fc_report(report_tensor)
        report_feature = self.relu(report_feature)

        wsi_feature = self.fc_wsi(wsi_tensor)
        wsi_feature = self.relu(wsi_feature)

        gene_feature = self.hidden_gene(gene_cnv_feature)
        gene_feature = self.relu(gene_feature)

        cnv_feature = self.hidden_cnv(gene_cnv_feature)
        cnv_feature = self.relu(cnv_feature)

        all_feature = torch.stack([gene_feature, cnv_feature, report_feature, wsi_feature])
        all_feature = self.dropout(all_feature)

        # all_feature = all_feature.permute(1, 0, 2)
        # all_feature = self.flatten(all_feature)
        
        # out = self.lin(all_feature)
        # return out

        """
        start:  torch.Size([4, 32, 64])
        before enccode:  torch.Size([32, 4, 64])
        after enccode:  torch.Size([32, 4, 64])
        end:  torch.Size([4, 32, 64])
        """
        all_feature = all_feature.permute(1,0,2)
        all_feature = self.encoder(all_feature)

        all_feature = all_feature.permute(1, 0, 2)
        all_feature = self.relu(all_feature)

        all_feature = self.dropout(all_feature)

        all_feature = all_feature.permute(1, 0, 2)
        all_feature = self.flatten(all_feature)
        
        out = self.lin(all_feature)
        return out

In [11]:
print(f"""
gene 数据维度:   {gene_dim}
cnv 数据维度:    {cnv_dim}
wsi 数据维度:    {wsi_dim}
report 数据维度: {report_dim}
""")

model = MultiOmicsModel().to(device)
# model(expr, cnv, report, sis)

# learning_rate = 0.001
# criterion = nn.BCEWithLogitsLoss()
# optimizer = optim.Adam(model.parameters(), lr=learning_rate)


# 定义损失函数和优化器
criterion = nn.CrossEntropyLoss()  # 适用于二分类问题
optimizer = optim.Adam(model.parameters(), lr=0.0001)



gene 数据维度:   2340
cnv 数据维度:    2340
wsi 数据维度:    2048
report 数据维度: 768



In [12]:
from sklearn.metrics import roc_auc_score

In [13]:
# 初始化TensorBoard SummaryWriter
writer = SummaryWriter(data_output_dir_path / 'runs/multi-model/attention')

num_epochs = 100
for epoch in range(num_epochs):

    model.train()
    # 导入 batch 数据
    for batch in train_loader:
        gene_tensor = batch["gene_tensor"].to(device)
        cnv_tensor = batch["cnv_tensor"].to(device)
        report_tensor = batch["report_tensor"].to(device)
        wsi_tensor = batch["wsi_tensor"].to(device)
        label_tensor = torch.squeeze(batch["label_tensor"]).to(torch.float32).to(device).view(-1, 1)
        label_tensor = label_tensor.squeeze()
        label_tensor = label_tensor.long()
        
        outputs = model(gene_tensor, cnv_tensor, report_tensor, wsi_tensor)

        optimizer.zero_grad()
        loss = criterion(outputs, label_tensor)
        loss.backward()
        optimizer.step()

    print("train loss: ", loss.item())
    writer.add_scalar('training_loss', loss.item(), epoch)

    correct = 0
    total = 0
    # 初始化变量来存储所有预测值和标签
    all_labels = []
    all_preds_prob = []

    # 在每个epoch结束时进行验证
    model.eval()
    with torch.no_grad():
        for batch in val_loader:
            gene_tensor = batch["gene_tensor"].to(device)
            cnv_tensor = batch["cnv_tensor"].to(device)
            report_tensor = batch["report_tensor"].to(device)
            wsi_tensor = batch["wsi_tensor"].to(device)
            label_tensor = torch.squeeze(batch["label_tensor"]).to(torch.float32).to(device).view(-1, 1)
            label_tensor = label_tensor.squeeze()
            label_tensor = label_tensor.long()

            outputs = model(gene_tensor, cnv_tensor, report_tensor, wsi_tensor)
            # outputs = model(gene_tensor)
            _, preds = torch.max(outputs, 1)  # 获取预测的类别
            total += label_tensor.size(0)
            correct += (preds == label_tensor).sum().item()
            
            # 假设outputs是模型的输出，形状为[24, 2]
            # 我们只关心正类的概率，所以取第二列（索引为1）
            all_preds_prob.extend(torch.sigmoid(outputs)[:, 1].cpu().numpy())
            all_labels.extend(label_tensor.cpu().numpy())

    # 计算准确率
    accuracy = correct / total
    print(f"Validation Accuracy: ({accuracy})")
    writer.add_scalar('validation_accuracy', accuracy, epoch)

    # 计算AUC
    auc = roc_auc_score(all_labels, all_preds_prob)
    print(f'AUC: {auc}')
    writer.add_scalar('validation_auc', auc, epoch)

writer.close()

!scp -r /workspace/pyfaster/examples/model/data/output/runs 0ba05b34d5857197e9c8ddb20dcf936d4b99996b-woplfq@woplfq.ssh.ide.cloud.tencent.com:/tmp ;
!rm -rf /workspace/pyfaster/examples/model/data/output/runs


train loss:  0.86210697889328
Validation Accuracy: (0.6513157894736842)
AUC: 0.7725069252077562
train loss:  0.6493523716926575
Validation Accuracy: (0.6776315789473685)
AUC: 0.8012465373961218
train loss:  0.744566023349762
Validation Accuracy: (0.7105263157894737)
AUC: 0.8438365650969528
train loss:  0.5345273613929749
Validation Accuracy: (0.7368421052631579)
AUC: 0.8772506925207757
train loss:  0.5699759125709534
Validation Accuracy: (0.8092105263157895)
AUC: 0.9210526315789473
train loss:  0.4824914038181305
Validation Accuracy: (0.9078947368421053)
AUC: 0.9560249307479224
train loss:  0.48598670959472656
Validation Accuracy: (0.9276315789473685)
AUC: 0.9776662049861495
train loss:  0.4938722848892212
Validation Accuracy: (0.9078947368421053)
AUC: 0.9823407202216066
train loss:  0.3618218004703522
Validation Accuracy: (0.9539473684210527)
AUC: 0.9882271468144044
train loss:  0.35057082772254944
Validation Accuracy: (0.9276315789473685)
AUC: 0.9906509695290858
train loss:  0.291845

In [14]:
preds

tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
       device='cuda:0')